# MADRID CSV

In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo de gráficos
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

### Leer el csv y sacar por pantalla las cinco primeras filas.

In [16]:
df = pd.read_csv("data/madrid_airbnb.csv")
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,6369,"Rooftop terrace room , ensuite bathroom",13660,Simon,Chamartín,Hispanoamérica,40.45724,-3.67688,Private room,60,1,78,2020-09-20,0.58,1,180
1,21853,Bright and airy room,83531,Abdel,Latina,Cármenes,40.40381,-3.74130,Private room,31,4,33,2018-07-15,0.42,2,364
2,23001,Apartmento Arganzuela- Madrid Rio,82175,Jesus,Arganzuela,Legazpi,40.38840,-3.69511,Entire home/apt,50,15,0,NaN,NaN,7,1
3,24805,Gran Via Studio Madrid,346366726,A,Centro,Universidad,40.42183,-3.70529,Entire home/apt,92,5,10,2020-03-01,0.13,1,72
4,26825,Single Room whith private Bathroom,114340,Agustina,Arganzuela,Legazpi,40.38975,-3.69018,Private room,26,2,149,2020-03-12,1.12,1,365


### Descripción del conjunto de datos, estándar

In [17]:
df.describe(include='all')

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
count,1.961800e+04,19615,1.961800e+04,19091,19618,19618,19618.000000,19618.000000,19618,19618.000000,19618.000000,19618.000000,13981,13981.000000,19618.000000,19618.000000
unique,NaN,18793,NaN,3900,21,128,NaN,NaN,4,NaN,NaN,NaN,1590,NaN,NaN,NaN
top,NaN,Habitación privada,NaN,Daniel,Centro,Embajadores,NaN,NaN,Entire home/apt,NaN,NaN,NaN,2020-03-01,NaN,NaN,NaN
freq,NaN,22,NaN,255,8649,2318,NaN,NaN,11314,NaN,NaN,NaN,359,NaN,NaN,NaN
mean,2.912200e+07,NaN,1.312165e+08,NaN,NaN,NaN,40.420984,-3.694040,NaN,129.271740,6.586196,31.858803,NaN,1.125958,10.229177,159.098328
std,1.351839e+07,NaN,1.166790e+08,NaN,NaN,NaN,0.022627,0.028671,NaN,484.143545,33.286582,63.938997,NaN,1.348235,23.546472,144.252803
min,6.369000e+03,NaN,7.952000e+03,NaN,NaN,NaN,40.332210,-3.863910,NaN,0.000000,1.000000,0.000000,NaN,0.010000,1.000000,0.000000
25%,1.903424e+07,NaN,2.765313e+07,NaN,NaN,NaN,40.409393,-3.707700,NaN,35.000000,1.000000,0.000000,NaN,0.170000,1.000000,0.000000
50%,3.187506e+07,NaN,9.901898e+07,NaN,NaN,NaN,40.419735,-3.701120,NaN,58.000000,2.000000,4.000000,NaN,0.590000,2.000000,126.000000
75%,4.090994e+07,NaN,2.256898e+08,NaN,NaN,NaN,40.430290,-3.685420,NaN,100.000000,3.000000,31.000000,NaN,1.630000,6.000000,320.000000


### Información sobre el tipo de datos de cada feature

In [18]:
print("--- Información de tipos de datos ---")
df.info()

--- Información de tipos de datos ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19618 entries, 0 to 19617
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              19618 non-null  int64  
 1   name                            19615 non-null  object 
 2   host_id                         19618 non-null  int64  
 3   host_name                       19091 non-null  object 
 4   neighbourhood_group             19618 non-null  object 
 5   neighbourhood                   19618 non-null  object 
 6   latitude                        19618 non-null  float64
 7   longitude                       19618 non-null  float64
 8   room_type                       19618 non-null  object 
 9   price                           19618 non-null  int64  
 10  minimum_nights                  19618 non-null  int64  
 11  number_of_reviews               19618 non-null  int64  

dataset de AirBnB con 19,618 propiedades y 15 columnas

### Contar los nulos por variable

In [19]:
print("--- Valores nulos por columna ---")
null_counts = df.isnull().sum()
print(null_counts)
if null_counts.sum() > 0:
    print("\nColumnas con valores nulos:")
    print(null_counts[null_counts > 0])
else:
    print("✓ No hay valores nulos en el dataset")

--- Valores nulos por columna ---
id                                   0
name                                 3
host_id                              0
host_name                          527
neighbourhood_group                  0
neighbourhood                        0
latitude                             0
longitude                            0
room_type                            0
price                                0
minimum_nights                       0
number_of_reviews                    0
last_review                       5637
reviews_per_month                 5637
calculated_host_listings_count       0
availability_365                     0
dtype: int64

Columnas con valores nulos:
name                    3
host_name             527
last_review          5637
reviews_per_month    5637
dtype: int64


#### Buscar valores extraños. Ver los valores únicos en cada feature

In [20]:
unique_values = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_values
unique_vals = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_vals

,features,n_values
0,id,"[6369, 21853, 23001, 24805, 26825, 28200, 3032..."
1,name,"[Rooftop terrace room , ensuite bathroom, Bri..."
2,host_id,"[13660, 83531, 82175, 346366726, 114340, 13090..."
3,host_name,"[Simon, Abdel, Jesus, A, Agustina, Dana, Angel..."
4,neighbourhood_group,"[Chamartín, Latina, Arganzuela, Centro, Salama..."
5,neighbourhood,"[Hispanoamérica, Cármenes, Legazpi, Universida..."
6,latitude,"[40.45724, 40.40381, 40.3884, 40.42183, 40.389..."
7,longitude,"[-3.67688, -3.7413, -3.69511, -3.70529, -3.690..."
8,room_type,"[Private room, Entire home/apt, Shared room, H..."
9,price,"[60, 31, 50, 92, 26, 85, 65, 54, 1400, 79, 90,..."


In [ ]:
unique_values = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_values
unique_vals = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_vals

,features,n_values
0,id,"[6369, 21853, 23001, 24805, 26825, 28200, 3032..."
1,name,"[Rooftop terrace room , ensuite bathroom, Bri..."
2,host_id,"[13660, 83531, 82175, 346366726, 114340, 13090..."
3,host_name,"[Simon, Abdel, Jesus, A, Agustina, Dana, Angel..."
4,neighbourhood_group,"[Chamartín, Latina, Arganzuela, Centro, Salama..."
5,neighbourhood,"[Hispanoamérica, Cármenes, Legazpi, Universida..."
6,latitude,"[40.45724, 40.40381, 40.3884, 40.42183, 40.389..."
7,longitude,"[-3.67688, -3.7413, -3.69511, -3.70529, -3.690..."
8,room_type,"[Private room, Entire home/apt, Shared room, H..."
9,price,"[60, 31, 50, 92, 26, 85, 65, 54, 1400, 79, 90,..."


### Obtener dataframe con features y sus valores únicos

In [21]:
unique_values_info = []
for col in df.columns:
    unique_vals = df[col].unique()
    unique_values_info.append({
        'features': col,
        'n_values': len(unique_vals),
        'values': str(list(unique_vals)[:10])  # Mostrar solo los primeros 10
    })

unique_df = pd.DataFrame(unique_values_info)
print("--- Valores únicos por feature ---")
unique_df[['features', 'n_values']]

--- Valores únicos por feature ---


,features,n_values
0,id,19618
1,name,18794
2,host_id,11325
3,host_name,3901
4,neighbourhood_group,21
5,neighbourhood,128
6,latitude,7536
7,longitude,7595
8,room_type,4
9,price,544


#### Tratar aquellos valores que entendamos que sean nulos

In [22]:
# Buscar valores que podrían representar nulos (como '?')
print("--- Verificando valores extraños ---")
strange_values_found = False
for col in df.columns:
    unique_vals = df[col].unique()
    if '?' in unique_vals:
        count_question = (df[col] == 'last_review').sum()
        print(f" '{col}' contiene {count_question} valores '?' (posibles nulos)")
        strange_values_found = True

if not strange_values_found:
    print("✓ No se encontraron valores extraños aparentes")
    # Imputaciones: reemplazar '?' por moda o eliminar filas
    print("--- Tratamiento de valores faltantes ---")

    original_shape = df.shape
    
for col in df.columns:
    if '?' in df[col].values:
        print(f"Tratando '{col}'...")
        # Opción 1: Imputar con la moda
        df[col] = df[col].replace('?', np.nan)
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)
        print(f"  → Imputado con moda: {mode_value}")

print(f"✓ Dataset shape: {original_shape} → {df.shape}")


--- Verificando valores extraños ---
✓ No se encontraron valores extraños aparentes
--- Tratamiento de valores faltantes ---
✓ Dataset shape: (19618, 16) → (19618, 16)


#### Mirad cuántos valores hay en cada feature, ¿Todas las features aportan información? Si alguna no aporta información, eliminadla

In [23]:
# Eliminar features que no aportan información (solo un valor único)

print("--- Verificando features informativas ---")
single_value_cols = []
for col in df.columns:
    unique_count = df[col].nunique()
    if unique_count <= 1:
        single_value_cols.append(col)
        print(f"⚠️  '{col}' tiene solo {unique_count} valor único")

if single_value_cols:
    df = df.drop(columns=single_value_cols)
    print(f"✓ Eliminadas {len(single_value_cols)} columnas sin información")
else:
    print("✓ Todas las features aportan información")

print(f"Forma final del dataset: {df.shape}")

--- Verificando features informativas ---
✓ Todas las features aportan información
Forma final del dataset: (19618, 16)


#### Separar entre variables predictoras y variables a predecir

In [25]:

target_col = 'price' if 'price' in df.columns else df.columns[0]
y = df[target_col]
X = df.drop(columns=[target_col])

print(f"Variable objetivo: {target_col}")
print(f"Variables predictoras: {X.shape[1]} features")
print(f"\n--- Distribución de la variable objetivo ---")
print(y.value_counts())
print(f"\nPorcentajes:")
print(y.value_counts(normalize=True) * 100)

Variable objetivo: price
Variables predictoras: 15 features

--- Distribución de la variable objetivo ---
price
50      585
30      570
20      566
40      561
25      545
       ... 
825       1
1150      1
643       1
536       1
347       1
Name: count, Length: 544, dtype: int64

Porcentajes:
price
50      2.981955
30      2.905495
20      2.885106
40      2.859619
25      2.778061
          ...   
825     0.005097
1150    0.005097
643     0.005097
536     0.005097
347     0.005097
Name: proportion, Length: 544, dtype: float64
